# Lab 1 — All six tracks in one notebook
### EP0418 · Topic 2 · Week 2

This notebook runs the baseline of every starter pack in one place, so you can
compare the six tracks before choosing one. It does not replace the packs: once
you have chosen, open your track's own starter notebook, which has the plots and
the live-refresh cell.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

**Nothing to upload.** Select **Runtime > Run all**: the notebook fetches the six
datasets itself. Every
section ends with the line `RECORD THIS:` — that is the number that goes in the
Lab 1 sheet's table.

In [ ]:
import os, zipfile
import pandas as pd, numpy as np

if os.path.exists("lab1_data.zip"):
    zipfile.ZipFile("lab1_data.zip").extractall(".")

PACK = {"smart_money_ohlcv.csv": "track1_smart_money", "smart_building_sensors.csv": "track2_smart_building",
        "sg_rainfall_daily.csv": "track3_smart_farm", "sg_psi_hourly.csv": "track3_smart_farm",
        "sg_taxi_hourly.csv": "track4_smart_city", "smart_self_daily.csv": "track5_smart_self",
        "example_data.csv": "track6_free_form"}
RAW = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/starter_packs/"

def find(name):
    # beside the notebook (or unzipped), inside a track folder, or fetched from the course repository
    for cand in [name] + [os.path.join(d, name) for d in os.listdir(".") if os.path.isdir(d)]:
        if os.path.exists(cand):
            return cand
    try:
        import urllib.request
        urllib.request.urlretrieve(RAW + PACK[name] + "/" + name, name)
    except Exception:
        from google.colab import files; files.upload()
    return name

RECORD = {}
def record(track, label, value):
    RECORD[track] = (label, value)
    print(f"\nRECORD THIS:  {label} = {value}")

## Track 1 — Smart Money (STI ETF daily bars)

In [ ]:
df = pd.read_csv(find("smart_money_ohlcv.csv"), parse_dates=["Date"]).sort_values("Date")
print(f"rows {len(df):,}   period {df.Date.min().date()} -> {df.Date.max().date()}   cadence: trading days")
bh_total = df.Close.iloc[-1] / df.Close.iloc[0] - 1
print(f"buy-and-hold total return: {bh_total:+.1%}   (buy on day 1, sell on the last day, no trading)")
record("1 Smart Money", "buy-and-hold return", f"{bh_total:+.1%}")

## Track 2 — Smart Building (classroom sensors, every 5 minutes)

In [ ]:
b = pd.read_csv(find("smart_building_sensors.csv"), parse_dates=["timestamp"])
print(f"rows {len(b):,}   period {b.timestamp.min()} -> {b.timestamp.max()}   cadence: 5 minutes")
alert = b.co2_ppm > 1000
print(f"fixed-threshold alerts (CO2 > 1000 ppm): {alert.sum()} of {len(b):,} readings; "
      f"{(alert & (b.occupancy == 0)).sum()} fired while the room was empty")
record("2 Smart Building", "alerts from the fixed threshold", int(alert.sum()))

## Track 3 — Smart Farm (NEA daily rainfall)

In [ ]:
r = pd.read_csv(find("sg_rainfall_daily.csv"), parse_dates=["date"]).sort_values("date")
print(f"rows {len(r)}   period {r.date.min().date()} -> {r.date.max().date()}   cadence: daily")
needs = r.rain_mm_mean < 1.0                       # the field needs water
yesterday = needs.shift(1)
m = yesterday.notna()
correct = (yesterday[m].astype(bool) == needs[m]).mean()
print(f"policy 'water if yesterday needed water': right on {correct:.1%} of days "
      f"({int((needs[m] & ~yesterday[m].astype(bool)).sum())} missed waterings, "
      f"{int((~needs[m] & yesterday[m].astype(bool)).sum())} wasted)")
record("3 Smart Farm", "days 'yesterday's value' was right", f"{correct:.1%}")

## Track 4 — Smart City (taxi availability, about hourly)

In [ ]:
t = pd.read_csv(find("sg_taxi_hourly.csv"), parse_dates=["timestamp"])
print(f"rows {len(t):,}   period {t.timestamp.min()} -> {t.timestamp.max()}   cadence: about hourly (irregular)")
s = t.set_index("timestamp").taxi_count.resample("h").mean()
sn = s.shift(168); m = s.notna() & sn.notna()
mae = (s - sn)[m].abs().mean()
print(f"seasonal-naive MAE (same hour last week): {mae:,.1f} taxis   (mean count {s.mean():,.0f})")
record("4 Smart City", "seasonal-naive MAE, taxis", round(float(mae), 1))

## Track 5 — Smart Self (wearable, one row per day)

In [ ]:
w = pd.read_csv(find("smart_self_daily.csv"), parse_dates=["date"]).sort_values("date")
print(f"rows {len(w)}   period {w.date.min().date()} -> {w.date.max().date()}   cadence: daily "
      f"({(w.date.max() - w.date.min()).days + 1 - len(w)} dates absent)")
d = w.set_index("date").asfreq("D")
pred = d.steps.rolling(7).mean().shift(1); m = pred.notna() & d.steps.notna()
mae = (d.steps[m] - pred[m]).abs().mean()
print(f"rolling 7-day mean baseline: MAE {mae:,.0f} steps   (mean daily steps {d.steps.mean():,.0f})")
record("5 Smart Self", "rolling 7-day MAE, steps", int(round(mae)))

## Track 6 — Free-form (the example file; yours will differ)

In [ ]:
e = pd.read_csv(find("example_data.csv"), parse_dates=["timestamp"]).sort_values("timestamp")
print(f"rows {len(e)}   period {e.timestamp.min().date()} -> {e.timestamp.max().date()}")
x = e.set_index("timestamp")["value"]
for name, p in [("naive (last value)", x.shift(1)), ("seasonal-naive (7 back)", x.shift(7))]:
    m = p.notna() & x.notna()
    print(f"{name:26s} MAE {(x[m] - p[m]).abs().mean():.3f}")
m = x.shift(1).notna()
record("6 Free-form", "naive MAE on the example file", round(float((x[m] - x.shift(1)[m]).abs().mean()), 3))

## The six numbers for your sheet

Copy this table into the Lab 1 sheet. Then answer the question that actually
decides your choice: which of these would be hardest to beat, and why?

In [ ]:
print(f"{'Track':18s} {'Baseline':40s} {'Number'}")
for track, (label, value) in RECORD.items():
    print(f"{track:18s} {label:40s} {value}")